# Session 48: Building a Web App with Streamlit

Data Science with Python, Level 3, Stage 7: Deployment

## Learning Objectives

By the end of this session you will be able to:

1. Explain how a Streamlit app runs.
2. Show text, numbers and charts on a page.
3. Collect inputs with widgets.
4. Load a saved model once with caching.
5. Show a prediction with clear messages.
6. Run and test an app from Colab.

## Where We Are

In Session 47 you saved the apartment price pipeline and wrote `predict_price`. A buyer or an estate agent will not open a notebook to use it. They need a web page with a few inputs and a clear answer.

**Streamlit** turns a Python script into a web app. There is no HTML or JavaScript to learn: each `st.` call adds something to the page. Today you build the Price Predictor app step by step and run it from Colab. In Session 49 you will put it on the internet with its own link.

## 1. How Streamlit Works

| Call | What it shows | Returns |
| --- | --- | --- |
| st.title("...") | A large page title | nothing |
| st.write(...) | Text, numbers, DataFrames, charts | nothing |
| st.number_input(...) | A number box | the number |
| st.slider(...) | A slider | the chosen value |
| st.selectbox(...) | A drop-down list | the chosen option |
| st.checkbox(...) | A tick box | True or False |
| st.button(...) | A button | True in the run after a click |
| st.metric(...) | A big number with a label | nothing |

## 2. Running Streamlit in Colab

In [ ]:
!pip install -q streamlit

In [ ]:
%%writefile app.py
import streamlit as st

st.title("Price Predictor")
st.write("Estimate the price of an apartment in seconds.")

In [ ]:
!streamlit run app.py &> logs.txt &
!curl -s ipv4.icanhazip.com
!npx --yes localtunnel --port 8501

## 3. Building the Price Predictor

In [ ]:
%%writefile app.py
import streamlit as st

st.title("Price Predictor")
area = st.slider("Area (square metres)", 20, 250, 85)
rooms = st.number_input("Rooms", min_value=1, max_value=8, value=3)
district = st.selectbox("District", ["Hillview", "Northgate", "Old Town",
                                     "Riverside", "Westpark"])
elevator = st.checkbox("Elevator", value=True)
st.write("You chose", area, "m2,", rooms, "rooms in", district)

In [ ]:
@st.cache_resource
def load_model():
    return joblib.load("models/price_model.joblib")

model = load_model()

In [ ]:
if st.button("Estimate price"):
    row = pd.DataFrame([{"district": district, "area_m2": area,
                         "rooms": rooms, "floor": floor,
                         "building_age": age, "distance_center_km": distance,
                         "elevator": elevator, "parking": parking}])
    price = float(model.predict(row)[0])
    st.metric("Estimated price", f"{price:,.0f} thousand")
    st.caption(f"Typical error: about {MAE} thousand either way.")

In [ ]:
left, right = st.columns(2)
with left:
    area = st.slider("Area (square metres)", 20, 250, 85)
    rooms = st.number_input("Rooms", 1, 8, 3)
with right:
    floor = st.number_input("Floor", 0, 30, 2)
    age = st.slider("Building age (years)", 0, 120, 20)

## 4. The Complete App

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import streamlit as st

MAE = 10
DISTRICTS = ["Hillview", "Northgate", "Old Town", "Riverside", "Westpark"]

st.set_page_config(page_title="Price Predictor")
st.title("Price Predictor")
st.write("Estimate an apartment's price from its size, age and location.")

@st.cache_resource
def load_model():
    return joblib.load("models/price_model.joblib")

model = load_model()

left, right = st.columns(2)
with left:
    district = st.selectbox("District", DISTRICTS)
    area = st.slider("Area (square metres)", 20, 250, 85)
    rooms = st.number_input("Rooms", 1, 8, 3)
    floor = st.number_input("Floor", 0, 30, 2)
with right:
    age = st.slider("Building age (years)", 0, 120, 20)
    distance = st.slider("Distance to centre (km)", 0.0, 30.0, 3.5, step=0.5)
    elevator = st.checkbox("Elevator", value=True)
    parking = st.checkbox("Parking", value=False)

if st.button("Estimate price", type="primary"):
    if rooms > area / 10:
        st.error("That is too many rooms for this area. Please check the inputs.")
        st.stop()
    row = pd.DataFrame([{"district": district, "area_m2": area, "rooms": rooms,
                         "floor": floor, "building_age": age,
                         "distance_center_km": distance,
                         "elevator": elevator, "parking": parking}])
    price = float(model.predict(row)[0])
    st.metric("Estimated price", f"{price:,.0f} thousand")
    st.caption(f"Typical error: about {MAE} thousand either way. "
               "Based on 800 practice apartments; not a valuation.")

| Part | Lines | What it does |
| --- | --- | --- |
| 1. Imports and settings | joblib, pandas, streamlit; MAE; DISTRICTS | Everything the app needs, defined once |
| 2. Page and model | set_page_config, title, load_model | Page title and the cached model |
| 3. Inputs | Two columns of widgets | District, area, rooms, floor, age, distance, flags |
| 4. Prediction | Button, check, DataFrame, metric, caption | Validate, predict, show the honest answer |

The full file is in the lecture notebook. It needs models/price_model.joblib in a models folder next to app.py.

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Model not found | FileNotFoundError for the model file | Upload `models/price_model.joblib` next to app.py. |
| Slow app | Loading the model on every rerun | Use `@st.cache_resource`. |
| Lost result | Expecting a button to stay True | Buttons are True only in the rerun after a click. |
| Running with python | `python app.py` shows warnings, no page | Use `streamlit run app.py`. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Hello Streamlit

Write and run an app with a title, a sentence about you, and a slider from 0 to 100 whose value is shown below it with `st.write`.

In [ ]:
# Write your code below

### In-Class Exercise 2: Show Data and a Chart

Write an app that loads `apartments.csv`, shows the first rows with `st.dataframe`, lets the user pick a district with a selectbox, and shows the average price of that district with `st.metric` and a bar chart of prices by rooms with `st.bar_chart`.

In [ ]:
# Write your code below

### In-Class Exercise 3: Validation Messages

Add to the price app: an error if the area per room is below 10, a warning (`st.warning`) if the building is older than 80 years because the training data has few such flats, and `st.success` above the metric when everything is fine.

In [ ]:
# Write your code below

### In-Class Exercise 4: A Price Range

Instead of one number, show the estimate as a range with two metrics side by side in columns: `Low estimate` (price minus MAE) and `High estimate` (price plus MAE).

In [ ]:
# Write your code below

## Summary

- A Streamlit app is a Python script; `st.` calls draw the page.
- The script reruns from the top whenever an input changes.
- Widgets return values: sliders, number boxes, drop-downs, tick boxes.
- `@st.cache_resource` loads the model only once.
- Show the answer with `st.metric` and honest messages.
- In Colab: write with `%%writefile`, run with `streamlit run`, open a tunnel.

## What's Next

Your app runs, but only while your Colab session is open. Next session you will deploy it on Streamlit Community Cloud straight from GitHub, so anyone can use it with a link, and you will learn to read logs and fix deployment errors.